<a href="https://colab.research.google.com/github/anokhina-rgb/Google-Colabs/blob/main/Copy_of_Corpus_Visualizer_results_of_scraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- КРОК 1: ВСТАНОВЛЕННЯ БІБЛІОТЕК ---
import subprocess
import sys

def install_packages():
    packages = ['feedparser', 'simplemma', 'beautifulsoup4', 'pandas', 'matplotlib', 'requests']
    for package in packages:
        try:
            __import__(package)
        except ImportError:
            print(f"Встановлення {package}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", package])

install_packages()

# --- КРОК 2: ІМПОРТ ТА НАЛАШТУВАННЯ ---
import json
import pandas as pd
import matplotlib.pyplot as plt
import os
import re
import sqlite3
import feedparser
import requests
import shutil
import time
from bs4 import BeautifulSoup
from datetime import datetime
from collections import Counter
import simplemma
from xml.etree.ElementTree import Element, SubElement, tostring
from xml.dom import minidom

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DB_NAME = "comprehensive_corpus.db"
# Джерела новин
RSS_FEEDS = {
    'ukraine_politics': ('https://www.ukrinform.ua/rss/politics', 'uk'),
    'ukraine_economy': ('https://www.ukrinform.ua/rss/economy', 'uk'),
    'world_news_bbc': ('http://feeds.bbci.co.uk/news/rss.xml', 'en'),
    'science_nature': ('https://www.nature.com/nature.rss', 'en'),
    'spain_elpais': ('https://elpais.com/rss/elpais/inenglish.xml', 'en'),
    'germany_dw': ('https://www.dw.com/xml/rss-de-all', 'de')
}

# --- КРОК 3: ФУНКЦІЇ ОБРОБКИ ТА ФОРМАТУВАННЯ ---

def get_full_article_text(url):
    """Скрейпінг тексту з очисткою від реклами та скриптів."""
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, 'html.parser')
            for s in soup(['script', 'style', 'nav', 'header', 'footer', 'aside', 'form', 'button']):
                s.decompose()

            content_div = soup.find('article') or soup.find('main') or soup.find('div', class_=re.compile(r'article|content|story'))
            paragraphs = content_div.find_all('p') if content_div else soup.find_all('p')
            text = "\n\n".join([p.get_text().strip() for p in paragraphs if len(p.get_text().strip()) > 40])
            return text if len(text) > 300 else None
    except:
        return None

def process_text_detailed(text, lang):
    """Токенізація та лематизація тексту."""
    if not text: return []
    text = re.sub(r'[^\w\s\.\,\!\?\-\"\']', ' ', text)
    tokens = simplemma.tokenizer.simple_tokenizer(text)
    processed = []
    for t in tokens:
        if t.strip() and not t.isdigit():
            lemma = simplemma.lemmatize(t, lang=lang)
            processed.append({'token': t, 'lemma': lemma})
    return processed

def create_tagged_xml(article_data, tokens):
    """Створення TEI XML структури."""
    root = Element('article')
    root.set('id', str(article_data['id']))
    root.set('lang', article_data['lang'])

    meta = SubElement(root, 'metadata')
    SubElement(meta, 'title').text = article_data['title']
    SubElement(meta, 'source').text = article_data['source']
    SubElement(meta, 'url').text = article_data['url']
    SubElement(meta, 'date').text = datetime.now().strftime("%Y-%m-%d")

    body = SubElement(root, 'body')
    for t in tokens:
        word = SubElement(body, 'w')
        word.set('lemma', t['lemma'])
        word.text = t['token']

    xml_str = minidom.parseString(tostring(root)).toprettyxml(indent="  ")
    return xml_str

def create_tei_json(article_data, tokens):
    """Генерація TEI у форматі JSON."""
    tei_dict = {
        "teiHeader": {
            "fileDesc": {
                "titleStmt": {"title": article_data['title']},
                "publicationStmt": {
                    "publisher": article_data['source'],
                    "date": datetime.now().strftime("%Y-%m-%d"),
                    "idno": article_data['id']
                },
                "sourceDesc": {"bibl": {"url": article_data['url']}}
            },
            "profileDesc": {"langUsage": {"language": {"ident": article_data['lang']}}}
        },
        "text": {
            "body": {
                "p": [{"w": t['token'], "lemma": t['lemma']} for t in tokens]
            }
        }
    }
    return json.dumps(tei_dict, ensure_ascii=False, indent=2)

# --- КРОК 4: ОСНОВНИЙ ЦИКЛ ---

def main():
    print("🚀 Оновлення бази даних та скрейпінг нових статей...")

    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS articles
                      (id INTEGER PRIMARY KEY,
                       title TEXT,
                       content TEXT,
                       source TEXT,
                       url TEXT UNIQUE,
                       lang TEXT,
                       category TEXT,
                       date_fetched TEXT,
                       tei_json TEXT)''')

    total_new = 0
    for cat, (url, lang) in RSS_FEEDS.items():
        print(f"📡 Перевірка RSS: {cat}...")
        feed = feedparser.parse(url)
        for entry in feed.entries[:50]:
            cursor.execute("SELECT id FROM articles WHERE url=?", (entry.link,))
            if not cursor.fetchone():
                content = get_full_article_text(entry.link)
                if content:
                    print(f"  📄 Додавання: {entry.title[:50]}...")
                    tokens = process_text_detailed(content, lang)
                    dummy_data = {'id': 'temp', 'title': entry.title, 'source': cat, 'url': entry.link, 'lang': lang}
                    t_json = create_tei_json(dummy_data, tokens)

                    cursor.execute("""INSERT INTO articles
                                   (title, content, source, url, lang, category, date_fetched, tei_json)
                                   VALUES (?,?,?,?,?,?,?,?)""",
                                 (entry.title, content, cat, entry.link, lang, cat, datetime.now().isoformat(), t_json))
                    total_new += 1
                    time.sleep(0.3)
    conn.commit()
    print(f"✅ Оновлення завершено. Додано {total_new} нових записів.")

    cursor.execute("SELECT * FROM articles")
    rows = cursor.fetchall()
    cols = [d[0] for d in cursor.description]
    data = [dict(zip(cols, row)) for row in rows]

    if not data:
        print("❌ База даних порожня.")
        return

    export_dir = "full_corpus_export"
    if os.path.exists(export_dir): shutil.rmtree(export_dir)

    for p in ['plain', 'vrt', 'tagged_xml', 'tei_json', 'stats']:
        os.makedirs(f"{export_dir}/{p}")

    all_lemmas = []

    print("⚙️ Експорт у всі формати (XML, VRT, TEI JSON)...")
    for item in data:
        tokens = process_text_detailed(item['content'], item['lang'])
        all_lemmas.extend([t['lemma'].lower() for t in tokens if len(t['lemma']) > 3])

        fname = f"id_{item['id']}"

        with open(f"{export_dir}/plain/{fname}.txt", "w", encoding="utf-8") as f:
            f.write(item['content'])

        with open(f"{export_dir}/vrt/{fname}.vrt", "w", encoding="utf-8") as f:
            f.write(f'<doc id="{item["id"]}" source="{item["source"]}">\n')
            for t in tokens: f.write(f"{t['token']}\t{t['lemma']}\n")
            f.write('</doc>\n')

        with open(f"{export_dir}/tagged_xml/{fname}.xml", "w", encoding="utf-8") as f:
            f.write(create_tagged_xml(item, tokens))

        with open(f"{export_dir}/tei_json/{fname}.json", "w", encoding="utf-8") as f:
            f.write(item['tei_json'] if item['tei_json'] else create_tei_json(item, tokens))

    # Візуалізація та Розширений Звіт
    print("📊 Генерація аналітики та детального звіту...")
    df = pd.DataFrame(data)

    plt.figure(figsize=(10, 5))
    df['lang'].value_counts().plot(kind='bar', color='teal', title='Мовний склад')
    plt.tight_layout()
    plt.savefig(f"{export_dir}/stats/languages.png")
    plt.close()

    # Створення текстового звіту з назвами та тематиками
    with open(f"{export_dir}/stats/report.txt", "w", encoding="utf-8") as f:
        f.write("=== ДЕТАЛЬНИЙ ЗВІТ КОРПУСУ ===\n")
        f.write(f"Дата звіту: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Загальна кількість документів у базі: {len(df)}\n")
        f.write(f"Унікальних лем (довжина > 3): {len(set(all_lemmas))}\n\n")

        f.write("СПИСОК СТАТЕЙ ТА ТЕМАТИКА:\n")
        f.write("-" * 50 + "\n")
        f.write(f"{'№':<4} | {'ТЕМАТИКА':<20} | {'НАЗВА СТАТТІ'}\n")
        f.write("-" * 50 + "\n")

        for idx, row in df.iterrows():
            # Скорочуємо назву для зручності у звіті, якщо вона задовга
            clean_title = row['title'].replace('\n', ' ').strip()
            f.write(f"{idx+1:<4} | {row['category']:<20} | {clean_title}\n")

        f.write("-" * 50 + "\n")

    # Архівування
    shutil.copy(DB_NAME, f"{export_dir}/{DB_NAME}")
    shutil.make_archive("corpus_package_full", 'zip', export_dir)

    print("📦 Архів corpus_package_full.zip готовий.")
    if IN_COLAB:
        files.download("corpus_package_full.zip")

if __name__ == "__main__":
    main()

Встановлення feedparser...
Встановлення simplemma...
Встановлення beautifulsoup4...
🚀 Оновлення бази даних та скрейпінг нових статей...
📡 Перевірка RSS: ukraine_politics...
📡 Перевірка RSS: ukraine_economy...
📡 Перевірка RSS: world_news_bbc...
  📄 Додавання: How rescue of US airman in remote part of Iran unf...
  📄 Додавання: The 40 minutes when the Artemis crew loses contact...
  📄 Додавання: Benefits and pensions rise as two-child cap ends...
  📄 Додавання: 'Two weeks will make such a difference': UK first ...
  📄 Додавання: Pepsi withdraws as UK festival sponsor after Kanye...
  📄 Додавання: Hungary alleges plot to blow up gas pipeline ahead...
  📄 Додавання: Disability benefits change means my son could lose...
  📄 Додавання: Family 'utterly devastated' after boy, 13, killed ...
  📄 Додавання: Seven arrested at RAF base demo accused of support...
  📄 Додавання: Royals attend Windsor Easter Sunday service...
  📄 Додавання: Oil prices choppy after expletive-laden Trump thre...
  📄 До

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>